1. Dueling DQN
2. n-step returns
3. Prioritized Experience Replay
5. Distributional DQN
5. Noisy Networks

In [ ]:
import os
import sys
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

if "google.colab" in sys.modules:
    archive_path = Path("/content/snake_dqn.zip")
    repository_path = Path("/content/snake_dqn")
    urlretrieve(
        "https://github.com/Mazako/snake_dqn/archive/refs/heads/master.zip",
        archive_path,
    )
    with ZipFile(archive_path) as archive:
        archive.extractall(repository_path)

    repository_root = next(path for path in repository_path.iterdir() if path.is_dir())
    os.chdir(repository_root)
    sys.path.insert(0, str(repository_root))

In [ ]:
from random import Random

import numpy as np
import torch
from torch import nn

from snake_dqn.agent import RelativeAction
from snake_dqn.replay_buffer import NStepReplayBuffer


In [ ]:
device_type = "cpu"
if torch.cuda.is_available():
    device_type = "cuda"
elif torch.mps.is_available():
    device_type = "mps"

device = torch.device(device_type)
print(f"{device=}")

In [ ]:
class DuelingDqn(nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, padding_mode="circular"),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, padding_mode="circular"),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, padding_mode="circular"),
            nn.ReLU(),
            nn.Flatten(),
        )

        self.value = nn.Sequential(
            nn.Linear(64 * 11 * 11, 256), nn.ReLU(), nn.Linear(256, 1)
        )

        self.advantage = nn.Sequential(
            nn.Linear(64 * 11 * 11, 256), nn.ReLU(), nn.Linear(256, 3)
        )

    def forward(self, x):
        base = self.network(x)
        value = self.value(base)
        advantage = self.advantage(base)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

In [ ]:
from collections import deque

from snake_dqn.double_dqn import select_actions, train_step
from snake_dqn.game import Game
from snake_dqn.multi_games_manager import MultiGamesManager


def evaluate(
    model: nn.Module,
    episodes: int,
    board_size: int,
    max_steps: int,
    device: torch.device,
    seed: int,
) -> float:
    was_training = model.training
    model.eval()
    scores = []
    rng = Random(seed)

    try:
        with torch.no_grad():
            for _ in range(episodes):
                game = Game(board_size, max_steps, seed=rng.getrandbits(64))
                state = game.state_img()

                while True:
                    action_index = model(state.to(device).unsqueeze(0)).argmax(dim=1).item()
                    _, _, done = game.step(RelativeAction(action_index))
                    state = game.state_img()

                    if done:
                        break

                scores.append(game.score)
    finally:
        model.train(was_training)

    return float(np.mean(scores))


def train(
    model: nn.Module,
    target_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    buffer: NStepReplayBuffer,
    rng: Random,
    seed: int,
    device: torch.device,
    episodes: int,
    board_size: int,
    max_steps: int,
    gamma: float,
    batch_size: int,
    epsilon: float,
    epsilon_min: float,
    epsilon_decay: float,
    num_envs: int,
    updates_per_batch: int,
    target_sync_interval: int,
    log_interval_iterations: int,
    eval_interval_iterations: int,
    eval_episodes: int,
    best_eval_score: float,
    checkpoint_path: Path,
) -> tuple[list[int], list[float], list[tuple[int, float]]]:
    environment_steps = 0
    next_target_sync_step = target_sync_interval
    completed_episodes = 0
    iteration = 0
    scores = []
    losses = []
    evaluations = []
    scores_avg = deque(maxlen=100)
    games = MultiGamesManager(num_envs, board_size, max_steps, seed=seed)
    while completed_episodes < episodes:
        actions = select_actions(model, torch.stack(games.states), epsilon, rng, device)

        transitions, completed_scores = games.step(
            actions, episodes - completed_episodes
        )
        for env_index, transition in enumerate(transitions):
            buffer.append(transition, env_index)
        environment_steps += len(transitions)
        for score in completed_scores:
            completed_episodes += 1
            scores.append(score)
            scores_avg.append(score)
            epsilon = max(epsilon_min, epsilon * epsilon_decay)
        for _ in range(updates_per_batch):
            loss = train_step(
                model,
                target_model,
                buffer,
                batch_size,
                gamma,
                loss_fn,
                optimizer,
                device,
                buffer.n_steps,
            )
            if loss is None:
                break
            losses.append(loss)
        iteration += 1

        if environment_steps >= next_target_sync_step:
            target_model.load_state_dict(model.state_dict())
            next_target_sync_step += target_sync_interval

        if iteration % log_interval_iterations == 0:
            mean_loss = np.mean(losses[-100:]) if losses else float("nan")
            mean_score = np.mean(scores_avg) if scores_avg else float("nan")
            last_score = scores[-1] if scores else "n/a"
            print(
                f"iteration={iteration} episodes={completed_episodes} "
                f"steps={environment_steps} score={last_score} "
                f"epsilon={epsilon:.3f} loss={mean_loss:.4f} "
                f"running avg={mean_score:.2f}"
            )

        if iteration % eval_interval_iterations == 0 or completed_episodes == episodes:
            eval_score = evaluate(
                model,
                eval_episodes,
                board_size,
                max_steps,
                device,
                seed + 1,
            )
            evaluations.append((iteration, eval_score))
            print(
                f"evaluation iteration={iteration} episodes={completed_episodes} "
                f"avg score={eval_score:.2f}"
            )
            if eval_score > best_eval_score:
                best_eval_score = eval_score
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "best_eval_score": best_eval_score,
                    },
                    checkpoint_path,
                )
                print(f"Saved best model with eval score={best_eval_score:.2f}")

    return scores, losses, evaluations


In [ ]:
import copy

seed = 42
rng = Random(seed)
torch.manual_seed(seed)
model = DuelingDqn().to(device)

checkpoint_path = Path("best_model_uniform_n_step.pt")
best_eval_score = float("-inf")
if checkpoint_path.is_file():
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    best_eval_score = float(checkpoint["best_eval_score"])
    print(f"Loaded model with best eval score={best_eval_score:.2f}")

target_model = copy.deepcopy(model).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.SmoothL1Loss()
gamma = 0.99
n_steps = 5
num_envs = 64
buffer = NStepReplayBuffer(100_000, num_envs, n_steps, gamma, seed=seed)

scores, losses, evaluations = train(
    model=model,
    target_model=target_model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    buffer=buffer,
    rng=rng,
    seed=seed,
    device=device,
    episodes=20_000,
    board_size=11,
    max_steps=500,
    gamma=gamma,
    batch_size=128,
    epsilon=0.2 if checkpoint_path.is_file() else 0.95,
    epsilon_min=0.01,
    epsilon_decay=0.9994,
    num_envs=num_envs,
    updates_per_batch=4,
    target_sync_interval=10_000,
    log_interval_iterations=10,
    eval_interval_iterations=500,
    eval_episodes=100,
    best_eval_score=best_eval_score,
    checkpoint_path=checkpoint_path,
)